# 📌 Recursive Abstractive Processing for Tree-Organized Retrieval (RAPTOR)

![Topic](https://img.shields.io/badge/Topic-RAPTOR-blue?style=flat-square)
![Category](https://img.shields.io/badge/Category-RAG-blueviolet?style=flat-square)
![Level](https://img.shields.io/badge/Level-Intermediate-yellow?style=flat-square)
![Last Updated](https://img.shields.io/badge/Updated-July%202026-blue?style=flat-square)
<br>
<br>
<br>
> <span style="font-size:20px;">**TL;DR** — **RAPTOR (Recursive Abstractive Processing for Tree-Organized Retrieval) is a retrieval technique for long documents. Instead of retrieving only small, isolated chunks of text like standard RAG, it builds a multi-level tree of summaries from the bottom up, then searches across all levels at once.** This lets a language model answer both detail-oriented and big-picture questions from the same document. The method works by recursively embedding, clustering, and summarizing chunks of text, constructing a tree with differing levels of summarization from the bottom up, then retrieving from this tree at inference time to integrate information across a document at different levels of abstraction.</span>

## Prerequisites

| Requirement | Details |
|-------------|---------|
| Python | 3.10+ |
| Libraries | `pip install numpy` |


---
## 1. Overview

<!-- What is this concept? 2–4 sentences that a newcomer could understand.
     Include: what problem it solves, why it exists, where it fits in the AI landscape. -->

Standard retrieval-augmented generation (RAG) splits a document into small chunks, embeds them, and retrieves the top-k chunks most similar to a query. This works fine for narrow factual questions, but it breaks down when a question requires understanding the document as a whole, since most existing methods retrieve only short contiguous chunks from a retrieval corpus, which limits holistic understanding of the overall document context.

RAPTOR, introduced by Parth Sarthi and coauthors (including Christopher Manning) at Stanford in January 2024, addresses this by not treating the corpus as a flat list of chunks. It organizes the chunks into a tree: the original passages sit at the leaves, and each level above them contains progressively more abstract summaries, up to a single root summary of the whole document.

---
## 2. How It Works

<!-- Break the concept into numbered steps or subsections.
     Use diagrams (images from assets/) where helpful.
     Each subsection should follow: description → danger/difficulty level → counter-technique or note -->

### 2.1 Chunk the text
The document is split into short passages, typically around 100 tokens each.

### 2.2 Embed the chunks
Each chunk is converted into a dense vector representation using SBERT (Sentence-BERT), a BERT-based encoder trained for sentence similarity. These become the leaf nodes of the tree.

### 2.3 Cluster the embeddings
This is the most technical step. The clustering algorithm is based on Gaussian Mixture Models (GMMs), which assume that data points are generated from a mixture of several Gaussian distributions, and it uses soft clustering, meaning it allows text segments to belong to multiple clusters at once, which helps capture varied thematic relevance. Because raw embeddings are high-dimensional and distances behave poorly at that scale, UMAP is used first to reduce dimensionality before the GMM is applied, and the Bayesian Information Criterion is used to automatically pick the right number of clusters.

### 2.4 Summarize each cluster
Each cluster of chunks is passed to a language model, which writes a concise summary. The original paper uses gpt-3.5-turbo for this summarization step, condensing a potentially large volume of retrieved information into a manageable size.

### 2.5 Recurse
Those new summaries become nodes themselves. Steps 2 to 4 repeat on them: embed the summaries, cluster them, summarize the clusters again, and so on, until you're left with a single root node representing the whole document.

### 2.6 Query the finished tree

At inference time, RAPTOR does not just search the leaves. It offers two retrieval strategies: Tree Traversal Retrieval, which starts at the root and walks down level by level comparing the query to nodes at each level, and Collapsed Tree Retrieval, which flattens the entire tree and searches all nodes at once regardless of level. The collapsed tree strategy evaluates all nodes simultaneously and retrieves at the correct granularity level, and the paper found it outperforms layer-by-layer traversal.

![RAPTOR.png](../assets/RAPTOR.png)

---
## 3. Advantages & Limitations

| | Aspect | Commentary |
|--|--------|------------|
| 🟢 | **Better long-document understanding** | Because retrieval can pull from summaries at any level, the model isn't limited to a handful of disconnected paragraphs when a question spans the whole document. |
| 🟢 | **Strong results on multi-step reasoning** | On question-answering tasks involving complex, multi-step reasoning, the paper reports state-of-the-art results, for example coupling RAPTOR retrieval with GPT-4 improved the best performance on the QUALITY benchmark by 20% in absolute accuracy. |
| 🟢 | **Model-agnostic gains** | RAPTOR consistently improves accuracy regardless of the underlying retriever, for instance combining it with BM25 raised performance from 23.52% to 27.93%, and combining it with DPR reached the highest score at 30.94%. |
| 🟢 | **Flexible retrieval granularity** | The two retrieval strategies (traversal vs collapsed) let you trade off between structured, level-aware search and simple, uniform search over everything. |
| 🟢 | **Works on top of existing embedders and retrievers** | It's an indexing and retrieval strategy layered on top of standard components (SBERT, BM25, DPR), not a replacement for them, so it's relatively easy to bolt onto an existing RAG pipeline. |
| 🔴 | **Expensive to build** | Every cluster at every level requires an LLM call to generate a summary, so indexing cost and time grow with document size and tree depth. |
| 🔴 | **Summarization errors can propagate** | If an early summary misrepresents its cluster, that error gets baked into every higher-level summary built on top of it. |
| 🔴 | **More moving parts** | Clustering (UMAP plus GMM plus BIC for cluster count) adds hyperparameters and complexity compared to just chunking and embedding. |
| 🔴 | **Static once built** | Adding or removing documents typically means rebuilding parts of the tree, which is less trivial than appending vectors to a flat index. |
| 🔴 | **Latency at query time** | Searching across multiple abstraction levels, or traversing the tree level by level, adds overhead compared to a single nearest-neighbor lookup over flat chunks. |

---
## 4. Code Example

> **Goal:** This is a simplified, illustrative sketch of the core loop. It is not a working library, just enough to show the shape of the algorithm.

In [2]:
import numpy as np

def embed(texts):
    # placeholder for a real SBERT call
    return np.random.rand(len(texts), 384)

def reduce_dimensions(embeddings, n_components=10):
    # placeholder for UMAP
    return embeddings[:, :n_components]

def cluster(embeddings):
    # placeholder for GMM + BIC cluster count selection
    n_clusters = max(1, len(embeddings) // 4)
    labels = np.random.randint(0, n_clusters, size=len(embeddings))
    return labels

def summarize(cluster_texts):
    # placeholder for an LLM summarization call
    joined = " ".join(cluster_texts)
    return f"Summary of {len(cluster_texts)} passages: {joined[:80]}..."

def build_raptor_tree(chunks, max_levels=4):
    tree = {"level_0": chunks}
    current_texts = chunks

    for level in range(1, max_levels + 1):
        if len(current_texts) <= 1:
            break

        embeddings = embed(current_texts)
        reduced = reduce_dimensions(embeddings)
        labels = cluster(reduced)

        summaries = []
        for label in set(labels):
            cluster_texts = [t for t, l in zip(current_texts, labels) if l == label]
            summaries.append(summarize(cluster_texts))

        tree[f"level_{level}"] = summaries
        current_texts = summaries

    return tree

def collapsed_tree_retrieval(query, tree, top_k=3):
    # search every node at every level at once
    all_nodes = [node for level in tree.values() for node in level]
    query_vec = embed([query])[0]
    node_vecs = embed(all_nodes)
    scores = node_vecs @ query_vec
    top_indices = np.argsort(scores)[-top_k:][::-1]
    return [all_nodes[i] for i in top_indices]

chunks = ["Passage 1 text...", "Passage 2 text...", "Passage 3 text...", "Passage 4 text..."]
tree = build_raptor_tree(chunks)
results = collapsed_tree_retrieval("What is the overall theme?", tree)
print(results)

['Summary of 4 passages: Passage 1 text... Passage 2 text... Passage 3 text... Passage 4 text......', 'Passage 3 text...', 'Passage 1 text...']


---
## 5. Key Takeaways
<div style="font-size: 16px; line-height: 1.6;">

- **RAPTOR replaces flat chunk retrieval with a tree of summaries.**
- **Clustering uses UMAP for dimensionality reduction and GMM for soft, probabilistic grouping.**
- **The tree is built bottom-up: chunks, then cluster summaries, then summaries of summaries.**
- **Collapsed-tree search outperformed level-by-level traversal in the original experiments.**
- **The main tradeoff is indexing cost and complexity in exchange for better multi-step reasoning.**

</div>